# Custom Image Dithering & Halftoning Generator
Load your own image and apply **Direct Thresholding**, **Ordered Dithering**, and **Floyd-Steinberg Diffusion**.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def load_and_preprocess_image(image_path: str, max_size: int = 512) -> np.ndarray:
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not load image at path: {image_path}")
    h, w = img.shape
    if max(h, w) > max_size:
        scale = max_size / float(max(h, w))
        new_w, new_h = int(w * scale), int(h * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32) / 255.0

def direct_threshold(image: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    return (image >= threshold).astype(np.float32)

def ordered_dither_bayer4x4(image: np.ndarray) -> np.ndarray:
    bayer_4x4 = (1.0 / 16.0) * np.array([
        [ 0,  8,  2, 10],
        [12,  4, 14,  6],
        [ 3, 11,  1,  9],
        [15,  7, 13,  5]
    ], dtype=np.float32)
    h, w = image.shape
    tiled_matrix = np.tile(bayer_4x4, (h // 4 + 1, w // 4 + 1))[:h, :w]
    return (image > tiled_matrix).astype(np.float32)

def floyd_steinberg_dither(image: np.ndarray) -> np.ndarray:
    img = image.copy().astype(np.float32)
    h, w = img.shape
    for y in range(h):
        for x in range(w):
            old_val = img[y, x]
            new_val = 1.0 if old_val >= 0.5 else 0.0
            img[y, x] = new_val
            err = old_val - new_val
            if x + 1 < w:
                img[y, x + 1] += err * (7.0 / 16.0)
            if x - 1 >= 0 and y + 1 < h:
                img[y + 1, x - 1] += err * (3.0 / 16.0)
            if y + 1 < h:
                img[y + 1, x] += err * (5.0 / 16.0)
            if x + 1 < w and y + 1 < h:
                img[y + 1, x + 1] += err * (1.0 / 16.0)
    return img

In [ ]:
# Set your file path here:
IMAGE_PATH = "your_image.png"

img = load_and_preprocess_image(IMAGE_PATH)
naive = direct_threshold(img)
bayer = ordered_dither_bayer4x4(img)
fs = floyd_steinberg_dither(img)

fig, axes = plt.subplots(1, 4, figsize=(20, 5), dpi=150)
axes[0].imshow(img, cmap='gray'); axes[0].set_title("Original")
axes[1].imshow(naive, cmap='gray'); axes[1].set_title("Direct Threshold")
axes[2].imshow(bayer, cmap='gray'); axes[2].set_title("Bayer Ordered Dither")
axes[3].imshow(fs, cmap='gray'); axes[3].set_title("Floyd-Steinberg Error Diffusion")
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()